Aayush kumar Bohara (15938004)

#Q1

In [3]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark session (if not already created)
spark = SparkSession.builder.appName("StudentDataAnalysis").getOrCreate()


In [4]:
# Load dataset
df = spark.read.csv("students_data.csv", header=True, inferSchema=True)

# Print schema
df.printSchema()


root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: double (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)



In [5]:
# Show first 5 rows
df.show(5)

+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|student_id|         name|gender|age|      department|year| gpa|scholarship|     city|               email|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
|       101| Aarav Sharma|  Male| 20|Computer Science|   2| 3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       102|  Priya Thapa|Female| 22|        Business|   4| 3.2|         No|  Pokhara| priya.thapa@uni.edu|
|       103|  Rohan Karki|  Male| 21|     Engineering|   3|NULL|         No| Lalitpur| rohan.karki@uni.edu|
|       104|     Sita Rai|Female| 19|Computer Science|   1| 3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       105|Bikash Gurung|  Male| 23|        Business|   4| 2.7|         No|   Butwal|bikash.gurung@uni...|
+----------+-------------+------+---+----------------+----+----+-----------+---------+--------------------+
only showing top 5 rows


#Q2

In [6]:
# Count rows
row_count = df.count()

# Count columns
column_count = len(df.columns)

print("Number of rows:", row_count)
print("Number of columns:", column_count)

Number of rows: 25
Number of columns: 10


#Q3

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType


In [8]:
# Define schema
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("department", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("gpa", FloatType(), True),
    StructField("scholarship", StringType(), True),
    StructField("city", StringType(), True),
    StructField("email", StringType(), True)
])

# Reload dataset with schema
df = spark.read.csv("students_data.csv", header=True, schema=schema)

# Print schema to verify
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- gpa: float (nullable = true)
 |-- scholarship: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)



#Q4

In [9]:
from pyspark.sql.functions import col, sum

In [10]:
# Count nulls in each column
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  2|         1|   0|  3|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+



#Q5

In [11]:
# Filter rows where GPA is null
df.filter(df.gpa.isNull()).show()

+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|student_id|          name|gender|age| department|year| gpa|scholarship|    city|               email|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+
|       103|   Rohan Karki|  Male| 21|Engineering|   3|NULL|         No|Lalitpur| rohan.karki@uni.edu|
|       110|  Nisha Tamang|Female| 20|       Arts|   2|NULL|         No|Lalitpur|nisha.tamang@uni.edu|
|       122|Dipika Niroula|Female| 20|   Business|   2|NULL|         No|  Dharan|dipika.niroula@un...|
+----------+--------------+------+---+-----------+----+----+-----------+--------+--------------------+



#Q6

In [12]:
from pyspark.sql.functions import avg

# Calculate mean GPA (ignoring nulls)
mean_gpa = df.select(avg("gpa")).collect()[0][0]

print("Mean GPA:", mean_gpa)

Mean GPA: 3.304545456712896


In [13]:
from pyspark.sql.functions import lit

# Fill null values
df = df.fillna({
    "gpa": mean_gpa,
    "department": "Undeclared",
    "age": 20
})

# Verify changes
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+----------+----+------+---+----------+----+---+-----------+----+-----+
|student_id|name|gender|age|department|year|gpa|scholarship|city|email|
+----------+----+------+---+----------+----+---+-----------+----+-----+
|         0|   0|     0|  0|         0|   0|  0|          0|   0|    0|
+----------+----+------+---+----------+----+---+-----------+----+-----+



#Q7

In [14]:
# Select specific columns
df.select("name", "department", "gpa").show()

+----------------+----------------+---------+
|            name|      department|      gpa|
+----------------+----------------+---------+
|    Aarav Sharma|Computer Science|      3.8|
|     Priya Thapa|        Business|      3.2|
|     Rohan Karki|     Engineering|3.3045454|
|        Sita Rai|Computer Science|      3.9|
|   Bikash Gurung|        Business|      2.7|
|  Anita Shrestha|     Engineering|      3.5|
|   Deepak Pandey|            Arts|      3.1|
|     Kavya Joshi|Computer Science|      3.7|
|    Suresh Limbu|     Engineering|      2.9|
|    Nisha Tamang|            Arts|3.3045454|
|   Prakash Bista|        Business|      3.4|
|     Manisha Oli|Computer Science|      3.6|
| Rajesh Adhikari|     Engineering|      3.0|
|    Sunita Magar|            Arts|      2.8|
|    Kiran Poudel|Computer Science|      3.3|
|     Rupa Basnet|      Undeclared|      3.7|
|  Anil Chaudhary|     Engineering|      2.6|
|    Puja Koirala|        Business|      3.5|
|     Nabin Dahal|            Arts

#Q8

In [15]:
df.filter(
    (df.department == "Computer Science") & (df.gpa > 3.5)
).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       101|    Aarav Sharma|  Male| 20|Computer Science|   2|3.8|        Yes|Kathmandu|aarav.sharma@uni.edu|
|       104|        Sita Rai|Female| 19|Computer Science|   1|3.9|        Yes|Kathmandu|    sita.rai@uni.edu|
|       108|     Kavya Joshi|Female| 20|Computer Science|   2|3.7|        Yes|  Pokhara| kavya.joshi@uni.edu|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



#Q9

In [16]:
df.filter(
    (df.year.isin(3, 4)) & (df.scholarship == "Yes")
).show()

+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|student_id|            name|gender|age|      department|year|gpa|scholarship|     city|               email|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+
|       111|   Prakash Bista|  Male| 24|        Business|   4|3.4|        Yes|Kathmandu|prakash.bista@uni...|
|       112|     Manisha Oli|Female| 21|Computer Science|   3|3.6|        Yes|  Pokhara| manisha.oli@uni.edu|
|       116|     Rupa Basnet|Female| 23|      Undeclared|   4|3.7|        Yes|  Pokhara| rupa.basnet@uni.edu|
|       120|Kabita Bhattarai|Female| 22|Computer Science|   4|3.9|        Yes|Bhaktapur|kabita.bhattarai@...|
|       121|      Suman Giri|  Male| 21|     Engineering|   3|3.4|        Yes|Kathmandu|  suman.giri@uni.edu|
+----------+----------------+------+---+----------------+----+---+-----------+---------+--------------------+



#Q10

In [17]:
from pyspark.sql.functions import when

In [18]:
df = df.withColumn(
    "grade",
    when(df.gpa >= 3.7, "Distinction")
    .when(df.gpa >= 3.0, "Merit")
    .otherwise("Pass")
)

df.select("name", "gpa", "grade").show()

+----------------+---------+-----------+
|            name|      gpa|      grade|
+----------------+---------+-----------+
|    Aarav Sharma|      3.8|Distinction|
|     Priya Thapa|      3.2|      Merit|
|     Rohan Karki|3.3045454|      Merit|
|        Sita Rai|      3.9|Distinction|
|   Bikash Gurung|      2.7|       Pass|
|  Anita Shrestha|      3.5|      Merit|
|   Deepak Pandey|      3.1|      Merit|
|     Kavya Joshi|      3.7|Distinction|
|    Suresh Limbu|      2.9|       Pass|
|    Nisha Tamang|3.3045454|      Merit|
|   Prakash Bista|      3.4|      Merit|
|     Manisha Oli|      3.6|      Merit|
| Rajesh Adhikari|      3.0|      Merit|
|    Sunita Magar|      2.8|       Pass|
|    Kiran Poudel|      3.3|      Merit|
|     Rupa Basnet|      3.7|Distinction|
|  Anil Chaudhary|      2.6|       Pass|
|    Puja Koirala|      3.5|      Merit|
|     Nabin Dahal|      3.2|      Merit|
|Kabita Bhattarai|      3.9|Distinction|
+----------------+---------+-----------+
only showing top

#Q11

In [19]:
df = df.drop("email", "student_id")

# Confirm columns
print(df.columns)

['name', 'gender', 'age', 'department', 'year', 'gpa', 'scholarship', 'city', 'grade']


#Q12

In [20]:
# Remove rows where department is null
df = df.dropna(subset=["department"])

# Remove duplicates based on name and department
df = df.dropDuplicates(["name", "department"])

# Verify
df.show()

+----------------+------+---+----------------+----+---------+-----------+---------+-----------+
|            name|gender|age|      department|year|      gpa|scholarship|     city|      grade|
+----------------+------+---+----------------+----+---------+-----------+---------+-----------+
|    Aarav Sharma|  Male| 20|Computer Science|   2|      3.8|        Yes|Kathmandu|Distinction|
|  Anil Chaudhary|  Male| 21|     Engineering|   3|      2.6|         No|Nepalgunj|       Pass|
|  Anita Shrestha|Female| 20|     Engineering|   2|      3.5|        Yes|Bhaktapur|      Merit|
|   Bikash Gurung|  Male| 23|        Business|   4|      2.7|         No|   Butwal|       Pass|
|  Bishal Ghimire|  Male| 23|     Engineering|   4|      2.8|         No|Kathmandu|       Pass|
|   Deepak Pandey|  Male| 22|            Arts|   3|      3.1|         No|Kathmandu|      Merit|
|  Dipika Niroula|Female| 20|        Business|   2|3.3045454|         No|   Dharan|      Merit|
|Kabita Bhattarai|Female| 22|Computer Sc

#Q13

In [21]:
df = df.withColumnRenamed("gpa", "grade_point_avg") \
       .withColumnRenamed("year", "study_year")

# Confirm changes
print(df.columns)

['name', 'gender', 'age', 'department', 'study_year', 'grade_point_avg', 'scholarship', 'city', 'grade']


#Q14

In [22]:
new_columns = [c.replace("_", " ") for c in df.columns]

df_spaces = df.toDF(*new_columns)

print(df_spaces.columns)

['name', 'gender', 'age', 'department', 'study year', 'grade point avg', 'scholarship', 'city', 'grade']


#Q15

In [24]:
df.select("grade_point_avg", "age").describe().show()

+-------+-------------------+------------------+
|summary|    grade_point_avg|               age|
+-------+-------------------+------------------+
|  count|                 25|                25|
|   mean|  3.304545450210571|              21.0|
| stddev|0.36909493967779344|1.3844373104863459|
|    min|                2.6|                19|
|    max|                3.9|                24|
+-------+-------------------+------------------+



#Q16

In [25]:
from pyspark.sql.functions import avg, col

df.groupBy("department") \
  .agg(
      avg("grade_point_avg").alias("avg_gpa")
  ) \
  .orderBy(col("avg_gpa").desc()) \
  .show()

+----------------+------------------+
|      department|           avg_gpa|
+----------------+------------------+
|      Undeclared| 3.700000047683716|
|Computer Science| 3.614285707473755|
|        Business|3.2209091186523438|
|            Arts|3.2009090423583983|
|     Engineering| 3.072077921458653|
+----------------+------------------+



#Q17

In [26]:
df.groupBy("gender", "scholarship") \
  .count() \
  .show()

+------+-----------+-----+
|gender|scholarship|count|
+------+-----------+-----+
|  Male|         No|   10|
|  Male|        Yes|    3|
|Female|         No|    5|
|Female|        Yes|    7|
+------+-----------+-----+



#Q18

In [27]:
from pyspark.sql.functions import count, avg, when

df.groupBy("city") \
  .agg(
      count("*").alias("student_count"),

      avg("grade_point_avg").alias("avg_gpa"),

      count(
          when(col("scholarship") == "Yes", True)
      ).alias("scholarship_holders")
  ) \
  .filter(col("student_count") > 2) \
  .show()

+---------+-------------+------------------+-------------------+
|     city|student_count|           avg_gpa|scholarship_holders|
+---------+-------------+------------------+-------------------+
|  Pokhara|            5|3.5599999904632567|                  4|
| Lalitpur|            4| 3.227272689342499|                  0|
|Kathmandu|            8|3.3375000059604645|                  4|
+---------+-------------+------------------+-------------------+



#Bonus Challenge

In [28]:
from pyspark.sql.functions import avg, when, col

# Reload original dataset first
df_bonus = spark.read.csv(
    "students_data.csv",
    header=True,
    inferSchema=True
)

In [29]:
# Drop rows where BOTH gpa and age are null
df_bonus = df_bonus.dropna(
    how="all",
    subset=["gpa", "age"]
)

In [30]:
# Calculate department average GPA
dept_avg = df_bonus.groupBy("department") \
    .agg(avg("gpa").alias("dept_avg_gpa"))

In [31]:
# Join averages
df_bonus = df_bonus.join(
    dept_avg,
    on="department",
    how="left"
)

In [32]:
# Fill null GPA with department average
df_bonus = df_bonus.withColumn(
    "gpa",
    when(
        col("gpa").isNull(),
        col("dept_avg_gpa")
    ).otherwise(col("gpa"))
)

In [33]:
# Add grade column
df_bonus = df_bonus.withColumn(
    "grade",
    when(col("gpa") >= 3.7, "Distinction")
    .when(col("gpa") >= 3.0, "Merit")
    .otherwise("Pass")
)

In [34]:
# Drop email column
df_bonus = df_bonus.drop("email")

# Rename gpa column
df_bonus = df_bonus.withColumnRenamed(
    "gpa",
    "grade_point_avg"
)

In [35]:
# Remove helper column
df_bonus = df_bonus.drop("dept_avg_gpa")

# Sort descending
df_bonus = df_bonus.orderBy(
    col("grade_point_avg").desc()
)


In [36]:
# Show result
df_bonus.show()

+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|      department|student_id|            name|gender| age|year|   grade_point_avg|scholarship|     city|      grade|
+----------------+----------+----------------+------+----+----+------------------+-----------+---------+-----------+
|Computer Science|       104|        Sita Rai|Female|  19|   1|               3.9|        Yes|Kathmandu|Distinction|
|Computer Science|       120|Kabita Bhattarai|Female|  22|   4|               3.9|        Yes|Bhaktapur|Distinction|
|Computer Science|       101|    Aarav Sharma|  Male|  20|   2|               3.8|        Yes|Kathmandu|Distinction|
|Computer Science|       108|     Kavya Joshi|Female|NULL|   2|               3.7|        Yes|  Pokhara|Distinction|
|            NULL|       116|     Rupa Basnet|Female|  23|   4|               3.7|        Yes|  Pokhara|Distinction|
|Computer Science|       112|     Manisha Oli|Female|  21|   3| 

In [37]:
# Save as parquet
df_bonus.write.mode("overwrite") \
    .parquet("students_parquet")